# Insider-Trading Detection — FINAL notebook (LR, HGB, Naive Bayes, Voting) with overfitting checks

**Upload to Colab (`/content`)**: `trade.csv` and `profile.csv` (any name starting with `trade` / `profile` works, e.g. `trade (3).csv`). Runtime: **CPU** (no GPU needed). Run all cells top to bottom (about 2–3 minutes).

**Design:** trader-wise split (a trader is never in both train and test) · the final model is fixed in advance (`Voting(LR+HGB)`) · the alert threshold is chosen on training out-of-fold predictions only · the test set is scored once · Cell 6 runs five separate overfitting checks.

In [ ]:
# CELL 1 — setup, config, load
import os, glob, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import HistGradientBoostingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score,
                             average_precision_score, confusion_matrix, precision_recall_curve)
warnings.filterwarnings("ignore")

# from google.colab import files; files.upload()     # <- run once to upload trade.csv and profile.csv

SEED, BETA, TEST_SIZE, N_FOLDS = 42, 2.0, 0.20, 5     # BETA>1 favours recall; use 1.0 for balanced precision/recall
FINAL_MODEL         = "Voting(LR+HGB)"                # fixed in advance: LR, HGB and Voting are statistically tied; Voting is never worse
USE_EVENT_TIMING    = True    # time-to-next-announcement features (retrospective surveillance)
USE_LINKED_ACCOUNTS = True    # trades by other accounts that share this trader's associated_pan
# For a real-time style detector set both to False.

def find(prefix):
    for d in ["/content", "."]:
        hits = sorted(glob.glob(f"{d}/{prefix}*.csv"), key=os.path.getmtime)
        if hits: return hits[-1]
    raise FileNotFoundError(f"No file starting with '{prefix}' found. Upload it to /content first.")

trade, profile = pd.read_csv(find("trade"), parse_dates=["timestamp"]), pd.read_csv(find("profile"))
print("trade:", trade.shape, "| profile:", profile.shape, "| insider share: %.1f%%" % (100 * trade.is_insider.mean()))
need_t = {"trade_id","trader_id","isin","timestamp","trade_type","quantity","total_value","volume_zscore","trade_frequency_7d",
          "order_size_percentile","time_to_next_event","pre_trade_return_1d","pre_trade_return_3d","pre_trade_return_5d","is_insider"}
need_p = {"trader_id","role_class","preclear_status","associated_pan","avg_trade_quantity","avg_trade_value"}
assert need_t <= set(trade.columns) and need_p <= set(profile.columns), "missing columns"
assert trade.trade_id.is_unique and set(trade.trader_id) <= set(profile.trader_id)

In [ ]:
# CELL 2 — features (point-in-time / retrospective only: no labels, no forward-looking prices)
t_feat = time.time()
df = trade.merge(profile[["trader_id","role_class","preclear_status","associated_pan","avg_trade_quantity","avg_trade_value"]],
                 on="trader_id", how="left", validate="many_to_one").reset_index(drop=True)

df["log_qty_ratio"]  = np.log(df.quantity    / df.avg_trade_quantity).clip(-3, 4)     # size vs trader's own baseline
df["log_val_ratio"]  = np.log(df.total_value / df.avg_trade_value).clip(-4, 5)
df["vol_zscore"]     = df.volume_zscore.clip(-4, 6)
df["order_size_pct"] = df.order_size_percentile
df["freq_7d"]        = df.trade_frequency_7d
df["preclear_req"]   = (df.preclear_status == "Required").astype(int)
for w in (1, 3, 5):                                                                    # momentum going INTO the trade
    df[f"pre_ret_{w}d"] = df[f"pre_trade_return_{w}d"].clip(-15, 15)
role = pd.get_dummies(df.role_class, prefix="role").drop(columns="role_Employee").astype(int)
df = pd.concat([df, role], axis=1)

FEATURES = ["log_qty_ratio","log_val_ratio","vol_zscore","order_size_pct","freq_7d","preclear_req",
            "pre_ret_1d","pre_ret_3d","pre_ret_5d"] + list(role.columns)

if USE_EVENT_TIMING:   # non-linear: insiders cluster in the last days/weeks, so use bins + capped log-hours
    df["log_hours_to_event"] = np.log1p(df.time_to_next_event.fillna(720).clip(upper=720))
    df["evt_within_3d"]  = (df.time_to_next_event <= 72).astype(int)
    df["evt_within_14d"] = (df.time_to_next_event <= 14 * 24).astype(int)
    FEATURES += ["log_hours_to_event", "evt_within_3d", "evt_within_14d"]

if USE_LINKED_ACCOUNTS:  # other accounts with the same PAN trading the same stock, same direction, within +-7 days
    out = np.zeros(len(df))
    shared = df.groupby("associated_pan").trader_id.transform("nunique") > 1
    for _, g in df[shared].groupby(["associated_pan", "isin", "trade_type"]):
        ts, tr, pos = g.timestamp.values, g.trader_id.values, g.index.values
        for i in range(len(g)):
            out[pos[i]] = ((np.abs((ts - ts[i]) / np.timedelta64(1, "D")) <= 7) & (tr != tr[i])).sum()
    df["linked_trades_7d"] = np.log1p(out)
    FEATURES.append("linked_trades_7d")

assert not ({"is_insider","trade_id","trader_id","isin","timestamp","price_return_1d","price_return_3d"} & set(FEATURES))
print(len(FEATURES), "features:", FEATURES)
print("feature engineering took %.1f s for %d trades" % (time.time() - t_feat, len(df)))

In [ ]:
# CELL 3 — split BY TRADER (a trader is never in both train and test). The test set stays locked until Cell 4 ends.
X, y, groups = df[FEATURES], df.is_insider.values, df.trader_id.values
tr_idx, te_idx = next(GroupShuffleSplit(1, test_size=TEST_SIZE, random_state=SEED).split(X, y, groups))
X_tr, y_tr, g_tr, X_te, y_te = X.iloc[tr_idx], y[tr_idx], groups[tr_idx], X.iloc[te_idx], y[te_idx]
assert not (set(g_tr) & set(groups[te_idx]))
print(f"train {len(y_tr)} trades ({y_tr.mean():.1%} insider) | test {len(y_te)} trades ({y_te.mean():.1%} insider)")

In [ ]:
# CELL 4 — compare models. Threshold uses ONLY out-of-fold predictions on the training set. Fit times are recorded.
def make_lr():  return make_pipeline(StandardScaler(), LogisticRegression(C=0.03, class_weight="balanced", max_iter=5000))
def make_hgb(): return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, l2_regularization=1.0,
                                                       min_samples_leaf=40, class_weight="balanced", random_state=SEED)
MODELS = {"LogisticRegression": make_lr, "HistGradBoosting": make_hgb, "GaussianNB": GaussianNB,
          "Voting(LR+HGB)": lambda: VotingClassifier([("lr", make_lr()), ("hgb", make_hgb())], voting="soft")}

def oof_proba(mk, Xa, ya, ga, n_folds=N_FOLDS):
    oof = np.zeros(len(ya))
    for a, b in GroupKFold(n_folds).split(Xa, ya, ga):
        oof[b] = mk().fit(Xa.iloc[a], ya[a]).predict_proba(Xa.iloc[b])[:, 1]
    return oof

def pick_threshold(yy, pr):        # centre of the plateau where F-beta is within 0.01 of its best (stable, not noise-fitted)
    grid = np.arange(0.05, 0.96, 0.01)
    f = np.array([fbeta_score(yy, (pr >= t).astype(int), beta=BETA, zero_division=0) for t in grid])
    return float(np.median(grid[f >= f.max() - 0.01]))

def metrics(yy, pr, thr):
    pred = (pr >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(yy, pred, labels=[0, 1]).ravel()
    return dict(precision=precision_score(yy, pred), recall=recall_score(yy, pred), f1=f1_score(yy, pred),
                f2=fbeta_score(yy, pred, beta=2), accuracy=accuracy_score(yy, pred), roc_auc=roc_auc_score(yy, pr),
                pr_auc=average_precision_score(yy, pr), tp=tp, fp=fp, fn=fn, tn=tn)

rows, fitted, oofs, fit_s, pred_s = [], {}, {}, {}, {}
for name, mk in MODELS.items():
    oofs[name] = oof_proba(mk, X_tr, y_tr, g_tr); thr = pick_threshold(y_tr, oofs[name])
    t0 = time.time(); fitted[name] = mk().fit(X_tr, y_tr); fit_s[name] = time.time() - t0
    t0 = time.time(); p_te_ = fitted[name].predict_proba(X_te)[:, 1]; pred_s[name] = time.time() - t0
    rows.append(dict(model=name, threshold=thr, oof_pr_auc=average_precision_score(y_tr, oofs[name]), **metrics(y_te, p_te_, thr)))
res = pd.DataFrame(rows).set_index("model")
BEST = FINAL_MODEL; assert BEST in MODELS
print("TEST-set results (all models, same split):")
print(res.drop(columns=["tp","fp","fn","tn"]).round(3).to_string())
print("\nSpeed (single fit on %d rows / predict %d rows), seconds:" % (len(y_tr), len(y_te)))
print(pd.DataFrame({"fit_s": fit_s, "predict_s": pred_s}).round(3).to_string())
print(f"\nFinal model (fixed in advance): {BEST}")

In [ ]:
# CELL 5 — final model report (test set scored once)
model, thr = fitted[BEST], float(res.loc[BEST, "threshold"])
p_tr, p_te = model.predict_proba(X_tr)[:, 1], model.predict_proba(X_te)[:, 1]
m = metrics(y_te, p_te, thr)
print(f"{BEST} | threshold {thr:.2f}")
print("\nTest confusion matrix   (rows = actual, cols = predicted)")
print(pd.DataFrame([[m["tn"], m["fp"]], [m["fn"], m["tp"]]], index=["genuine", "insider"], columns=["pred genuine", "pred insider"]))
print("precision %.3f | recall %.3f | F1 %.3f | accuracy %.3f | ROC-AUC %.3f | PR-AUC %.3f" %
      (m["precision"], m["recall"], m["f1"], m["accuracy"], m["roc_auc"], m["pr_auc"]))

pr, rc, _ = precision_recall_curve(y_te, p_te)
plt.figure(figsize=(5, 3.4)); plt.plot(rc, pr); plt.scatter([m["recall"]], [m["precision"]], c="r", label="chosen threshold")
plt.axhline(y_te.mean(), ls=":", c="grey", label="no-skill"); plt.xlabel("recall"); plt.ylabel("precision")
plt.title(f"Test precision-recall: {BEST}"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# CELL 6 — OVERFITTING CHECKS (five independent tests on the final model)
mk = MODELS[BEST]

# (a) train vs out-of-fold vs test: a big gap would mean memorising
tab = pd.DataFrame({"train": metrics(y_tr, p_tr, thr), "out-of-fold": metrics(y_tr, oofs[BEST], thr), "TEST": m})
print("(a) train / out-of-fold / test\n", tab.loc[["precision", "recall", "f1", "pr_auc", "roc_auc"]].round(3).to_string())
gap = tab.loc["pr_auc", "train"] - tab.loc["pr_auc", "TEST"]
print("    PR-AUC gap train-test = %.3f  ->  %s" % (gap, "small: no material overfitting" if gap < 0.03 else "WARNING: noticeable overfitting"))

# (b) 6 different trader splits (threshold re-chosen from each split's own training data)
def run_protocol(seed):
    a_, b_ = next(GroupShuffleSplit(1, test_size=TEST_SIZE, random_state=seed).split(X, y, groups))
    Xa, ya, ga = X.iloc[a_], y[a_], groups[a_]
    th = pick_threshold(ya, oof_proba(mk, Xa, ya, ga, n_folds=3)); mdl = mk().fit(Xa, ya)
    return metrics(y[b_], mdl.predict_proba(X.iloc[b_])[:, 1], th), metrics(ya, mdl.predict_proba(Xa)[:, 1], th)
runs = [run_protocol(s) for s in range(6)]
te_d, tr_d = pd.DataFrame([r[0] for r in runs]), pd.DataFrame([r[1] for r in runs])
print("\n(b) 6 repeated trader-wise splits (mean ± std on TEST):")
for k in ["precision", "recall", "f1", "accuracy", "pr_auc", "roc_auc"]:
    print(f"    {k:10s} {te_d[k].mean():.3f} ± {te_d[k].std():.3f}")
print("    mean train-test PR-AUC gap: %.3f" % (tr_d.pr_auc.mean() - te_d.pr_auc.mean()))

# (c) permutation test: shuffle ALL labels -> a model with no leak / no memorisation must score AUC ~0.50
rng = np.random.default_rng(SEED); aucs = []
for _ in range(5):
    yp = rng.permutation(y); aucs.append(roc_auc_score(yp[te_idx], mk().fit(X_tr, yp[tr_idx]).predict_proba(X_te)[:, 1]))
print("\n(c) shuffled-label test AUC: %.3f ± %.3f   (should be ~0.50)" % (np.mean(aucs), np.std(aucs)))

# (d) out-of-time: train on the earliest 75% of dates, test on the latest 25%
cut = df.timestamp.quantile(0.75); a_i, b_i = np.where(df.timestamp < cut)[0], np.where(df.timestamp >= cut)[0]
th_t = pick_threshold(y[a_i], oof_proba(mk, X.iloc[a_i], y[a_i], groups[a_i], n_folds=3))
oot = metrics(y[b_i], mk().fit(X.iloc[a_i], y[a_i]).predict_proba(X.iloc[b_i])[:, 1], th_t)
print("\n(d) out-of-time test (latest 25%% of dates): precision %.3f | recall %.3f | F1 %.3f | PR-AUC %.3f"
      % (oot["precision"], oot["recall"], oot["f1"], oot["pr_auc"]))

# (e) learning curve: if train and test scores converge as data grows, the model is not overfitting
traders = np.array(sorted(set(g_tr))); np.random.default_rng(SEED).shuffle(traders)
print("\n(e) learning curve (PR-AUC):   train-size   train   test")
for frac in (0.25, 0.5, 0.75, 1.0):
    keep = np.isin(g_tr, traders[: int(len(traders) * frac)]); mdl = mk().fit(X_tr[keep], y_tr[keep])
    print("                                %5.0f%% (%5d)  %.3f   %.3f" % (100 * frac, keep.sum(),
          average_precision_score(y_tr[keep], mdl.predict_proba(X_tr[keep])[:, 1]),
          average_precision_score(y_te, mdl.predict_proba(X_te)[:, 1])))

In [ ]:
# CELL 7 — what drives the alerts + save outputs
pi = permutation_importance(model, X_te, y_te, scoring="average_precision", n_repeats=3, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
print("Top features (drop in test PR-AUC when shuffled):"); print(imp.head(8).round(4).to_string())
imp.head(8).iloc[::-1].plot.barh(figsize=(6, 3.2), title="Permutation importance"); plt.tight_layout(); plt.show()

alerts = df.loc[X_te.index, ["trade_id", "trader_id", "role_class", "trade_type", "quantity"]].copy()
alerts["prob_insider"], alerts["flag"], alerts["label"] = p_te, (p_te >= thr).astype(int), y_te
alerts.sort_values("prob_insider", ascending=False).to_csv("test_alerts.csv", index=False)
joblib.dump({"model": model, "features": FEATURES, "threshold": thr, "name": BEST}, "insider_model.joblib")
print("\nsaved: test_alerts.csv, insider_model.joblib")
alerts.sort_values("prob_insider", ascending=False).head(8).round(3)

In [ ]:
# CELL 8 — reality check: precision falls as insider trading gets rarer (this data is ~17% insider)
tpr, fpr = m["recall"], m["fp"] / (m["fp"] + m["tn"])
for pi_ in [y.mean(), 0.05, 0.01]:
    print(f"if {pi_:5.1%} of trades were insider -> expected precision {tpr*pi_/(tpr*pi_+fpr*(1-pi_)):5.1%}")

### Why is it so fast?
20,000 rows × 18 features is a **tiny** table (about 360,000 numbers). Logistic Regression is a smooth convex problem solved in milliseconds; Naive Bayes only computes means and variances;
Histogram Gradient Boosting first bins every feature into 256 buckets, so each of its 200 shallow trees is cheap. Only trees + repeated validation take seconds. Nothing here needs a GPU.
Speed is not a sign of a mistake — Cell 6 (shuffled labels, out-of-time, learning curve) is what proves the model is not cheating.

### Read before quoting numbers
The data is **synthetic**: the generator planted the patterns this model finds (trades just before announcements, oversized trades, linked accounts), so real-world scores will be lower.
About 5% of true insider trades are labelled 0 by design, which caps the achievable score. Cell 8 shows why precision drops when insider trading is rare.